# Can the normalized proxy–judge gap be predicted?

This is a standalone go/no-go experiment performed **before another PPO run**. It uses 5,000 base-policy responses for frozen calibration and a separate 20,000 responses for GapFinder. The GapFinder data is split into 70% train, 15% validation, and 15% test.

The target is the signed normalized gap `d = proxy_z - judge_z`. All final metrics are reported only on the untouched test split.

In [ ]:
from dataclasses import replace
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import functions
from Models.model_gap_finder import GapFinder
from Trainers.trainer_gap_finder import (
    GapFinderTrainer,
    GapFinderTrainingConfig,
    compute_gap_finder_report,
)
from functions import ConfigTrainClassifier, PolicySpec, RewardSpec

logging.getLogger().setLevel(logging.INFO)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
DATASET_NAME = "Anthropic/hh-rlhf"
CALIBRATION_START = 0
CALIBRATION_END = 5_000
GAP_DATA_START = 5_000
GAP_DATA_END = 25_000  # exactly 20,000 different prompts
RANDOM_STATE = 42
OUTPUT_ROOT = Path("outputs/gap_finder_predictability")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# No LoRA is attached to the frozen base policy used to generate responses.
policy_spec = PolicySpec(
    model_name="Qwen/Qwen3-0.6B",
    lora_config=None,
)
proxy_spec = RewardSpec(
    class_name="RewardModel",
    model_name="Skywork/Skywork-Reward-V2-Qwen3-0.6B",
    mode_name="proxy",
)
judge_spec = RewardSpec(
    class_name="RewardModel",
    model_name="Skywork/Skywork-Reward-V2-Qwen3-4B",
    mode_name="judge",
)

# H200-oriented batch sizes. Reduce judge_batch_size first if needed.
calibration_config = ConfigTrainClassifier(
    dataset_name=DATASET_NAME,
    policy=policy_spec,
    reward=proxy_spec,
    judge=judge_spec,
    start_dataset=CALIBRATION_START,
    end_dataset=CALIBRATION_END,
    generation_batch_size=64,
    reward_batch_size=128,
    judge_batch_size=32,
    score_max_length=1024,
    gap_finder_batch_size=128,
    gap_finder_epochs=3.0,
)
gap_data_config = replace(
    calibration_config,
    start_dataset=GAP_DATA_START,
    end_dataset=GAP_DATA_END,
)

## 1. Build frozen calibration on a separate response set

In [ ]:
gap_calibration = functions.calculate_gap_calibration(calibration_config)
calibration_path = gap_calibration.save(OUTPUT_ROOT / "calibration.json")
gap_calibration

## 2. Generate and label 20,000 policy responses

This stage generates base-policy answers, scores every answer with proxy and judge sequentially, normalizes both scores using the frozen calibration, and stores `proxy_z - judge_z` as the regression label.

In [ ]:
gap_dataset = functions.collect_gap_finder_dataset(
    gap_data_config,
    gap_calibration,
)
assert len(gap_dataset) == 20_000

gap_dataset.save(OUTPUT_ROOT / "all_20k.json")
train_dataset, validation_dataset, test_dataset = gap_dataset.split_three_way(
    train_size=0.70,
    validation_size=0.15,
    test_size=0.15,
    random_state=RANDOM_STATE,
)
assert (len(train_dataset), len(validation_dataset), len(test_dataset)) == (14_000, 3_000, 3_000)

train_dataset.save(OUTPUT_ROOT / "train_70.json")
validation_dataset.save(OUTPUT_ROOT / "validation_15.json")
test_dataset.save(OUTPUT_ROOT / "test_15.json")
{"train": len(train_dataset), "validation": len(validation_dataset), "test": len(test_dataset)}

In [ ]:
all_gaps = np.asarray([row["labels"] for row in gap_dataset.dataset])
pd.Series(all_gaps).describe(percentiles=[0.50, 0.90, 0.95, 0.99]).to_frame("normalized_gap"), {
    "theta": gap_calibration.theta,
    "count_d_gt_theta": int((all_gaps > gap_calibration.theta).sum()),
    "fraction_d_gt_theta": float((all_gaps > gap_calibration.theta).mean()),
}

## 3. Train GapFinder with validation-based checkpoint selection

In [ ]:
run_directory = OUTPUT_ROOT / f"id={gap_dataset.id}"
gap_finder = GapFinder(
    gap_data_config.gap_finder_model_name,
    gap_finder_id=gap_dataset.id,
    source_policy=policy_spec.model_name,
)
training_config = GapFinderTrainingConfig(
    output_dir=str(run_directory),
    epochs=gap_data_config.gap_finder_epochs,
    batch_size=gap_data_config.gap_finder_batch_size,
    learning_rate=gap_data_config.gap_finder_learning_rate,
    max_length=gap_data_config.gap_finder_max_length,
    lora_settings=gap_data_config.gap_finder_lora_settings,
)
gap_trainer = GapFinderTrainer(gap_finder, training_config)
hf_trainer = gap_trainer.train(train_dataset, validation_dataset)
validation_metrics = gap_trainer.evaluate(hf_trainer, validation_dataset)
validation_metrics

## 4. Final untouched test-set report

`MAE d>theta` is calculated only on examples whose true gap exceeds the frozen calibration threshold. Detector metrics compare `predicted_gap > theta` with `actual_gap > theta`.

In [ ]:
test_prompts = [row["prompt"] for row in test_dataset.dataset]
test_answers = [row["answer"] for row in test_dataset.dataset]
actual_test_gaps = np.asarray([row["labels"] for row in test_dataset.dataset])
predicted_test_gaps = np.asarray(
    gap_finder.predict_gap(
        test_prompts,
        test_answers,
        batch_size=gap_data_config.gap_finder_batch_size,
        max_length=gap_data_config.gap_finder_max_length,
    )
)

test_report = compute_gap_finder_report(
    actual_test_gaps,
    predicted_test_gaps,
    theta=gap_calibration.theta,
)
zero_baseline_report = compute_gap_finder_report(
    actual_test_gaps,
    np.zeros_like(actual_test_gaps),
    theta=gap_calibration.theta,
)
test_report["mse_improvement_over_zero"] = (
    1.0 - test_report["mse"] / zero_baseline_report["mse"]
)
test_report["mae_improvement_over_zero"] = (
    1.0 - test_report["mae"] / zero_baseline_report["mae"]
)

report_path = run_directory / "test_report.json"
report_path.write_text(json.dumps(test_report, indent=2, sort_keys=True) + "\n")
pd.DataFrame([test_report]).T.rename(columns={0: "test_value"})

In [ ]:
comparison = pd.DataFrame(
    {
        "zero_prediction_baseline": zero_baseline_report,
        "gap_finder": test_report,
    }
)
comparison.loc[[
    "mse",
    "mae",
    "r2",
    "pearson",
    "spearman",
    "mae_d_gt_theta",
    "precision_d_gt_theta",
    "recall_d_gt_theta",
    "f1_d_gt_theta",
]]

## 5. Inspect the largest true-gap test examples

In [ ]:
test_examples = pd.DataFrame(
    {
        "prompt": test_prompts,
        "answer": test_answers,
        "actual_gap": actual_test_gaps,
        "predicted_gap": predicted_test_gaps,
        "absolute_error": np.abs(actual_test_gaps - predicted_test_gaps),
        "actual_detector": actual_test_gaps > gap_calibration.theta,
        "predicted_detector": predicted_test_gaps > gap_calibration.theta,
    }
)
test_examples.sort_values("actual_gap", ascending=False).head(20)